In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

import anndata as ad
import colorcet as cc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
from rich import print

In [3]:
from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class PARAMS:
    """set all params for our analysis"""

    # DATA_FOLDER: str = (
    #     "/home/labs/nyosef/Collaboration/SpatialDatasets/xenium/human_breast_cancer/"
    # )
    DATA_FOLDER: str = "/home/nathanl/Data/"
    DATA_FILE: str = "proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2.h5ad"

    FIGURES_FOLDER: str = "figures/"

    # for the data
    COORDS: str = "spatial"
    BATCH: str = "sample"
    SAMPLE: str = "sample"
    CELL_TYPE: str = "predictions_resolvi_proseg_coarse_corrected"
    NICHE: str = "kmeans_alpha_5"

    # for scVI
    EXPRESSION_MODEL = "scanvi"
    N_LAYERS: int = 1
    N_LATENT: int = 10
    LIKELIHOOD: str = "poisson"
    ######################
    N_EPOCHS_SCVI: int = 1001
    N_EPOCHS_RESOLVI: int = 100
    LR_SCVI: float = 5e-4
    BATCH_SIZE_SCVI: int = 512
    OPTIMIZER: str = "Adam"
    ######################
    WEIGHT_DECAY: float = 1e-6
    KL_WARMUP: Optional[int] = 400
    MAX_KL_WEIGHT: float = 1

    # for nicheVI
    K_NN: int = 20  # TODO try also less cells per niche, like 12
    N_LAYERS_NICHE: int = 1
    N_LAYERS_COMPO: int = 1
    N_HIDDEN: int = 128
    N_HIDDEN_COMPO: int = 128  # TODO make it adaptable
    N_HIDDEN_NICHE: int = 128
    N_LATENT_NICHEVI: int = 10
    ######################
    N_EPOCHS_NICHEVI: int = 1001
    LR_NICHEVI: float = 1e-3
    BATCH_SIZE_NICHEVI: int = 512
    REDUCE_LR_ON_PLATEAU: bool = True
    SPATIAL_WARMUP: int | None = None
    MIN_SPATIAL_WEIGHT: float = 1.0 if SPATIAL_WARMUP is None else 0
    MAX_SPATIAL_WEIGHT: float = 1.0
    USE_LAYER_NORM: bool = True
    USE_BATCH_NORM: bool = False


setup = PARAMS()

In [4]:
sc.set_figure_params(dpi=150, fontsize=12, format="svg")
figure_dir = setup.FIGURES_FOLDER

plots = True

In [5]:
scvi.settings.seed = 34

Seed set to 34


In [6]:
data_dir, data_file = setup.DATA_FOLDER, setup.DATA_FILE

data_file_name = os.path.splitext(data_file)[0]
print(data_file_name)

path_to_save = os.path.join("../niche-VI-experiments/checkpoints", data_file_name)
os.makedirs(path_to_save, exist_ok=True)

proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2

In [7]:
# adata = ad.read_h5ad(data_dir + data_file)
adata = ad.read_h5ad(path_to_save + "/proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2_nicheVI.h5ad")

del adata.uns["log1p"]

print(adata)

AnnData object with n_obs × n_vars = 278658 × 313
    obs: 'sample', '_scvi_batch', 'cell_type', '_scvi_labels', 'batch', 'predictions_scanvi', 'x', 'y', 'z', 
'observed_x', 'observed_y', 'observed_z', 'fov', 'probability', 'background', 'confusion', 'n_counts', '_indices', 
'_scvi_ind_x', 'predicted_celltype_resolvi', 'predicted_celltype_prob_resolvi', 'predictions_resolvi_proseg', 
'predictions_resolvi_proseg_coarse', 'predicted_celltype_resolvi_coarse', 'index', 'kmeans_alpha_4', 
'kmeans_alpha_5', 'kmeans_alpha_5_transferred', 'predictions_resolvi_proseg_coarse_corrected', '_scvi_sample'
    var: 'mt'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'pca'
    obsm: 'X_banksy_02_harmony', 'X_pca', 'X_resolvi', 'X_resolvi_MDE', 'X_resolvi_proseg', 'X_scANVI', 
'X_scANVI_MDE', 'X_scanvi', 'X_spatial', 'banksy_pc_20_lambda_0.0', 'banksy_pc_20_lambda_0.2', 
'celltype_transfer_resolvi', 'distance_neighbor', 'index_neighbor', 
'ln_s10_w400_scanvi_lr0.0005_poisson_X_nicheVI', 'ln_s20_w400_scanvi_lr0.0005_poisson_X_nicheVI', 
'mog_ln_s20_w400_scanvi_lr0.0005_poisson_X_nicheVI', 'neighborhood_composition', 'niche_composition', 
'niche_distances', 'niche_indexes', 'observed_spatial', 'qz1_m', 'qz1_m_niche_ct', 'spatial'
    varm: 'PCs'
    layers: 'counts', 'counts_wo_bg'

In [8]:
# adata_CD8 = adata[adata.obs[setup.CELL_TYPE].isin(["CD8+_T_Cells"])].copy()
# adata_CD8.obsm["MDE_CD8+_T_Cells_nichevi"] = scvi.model.utils.mde(
#     adata_CD8.obsm[model+"_X_nicheVI"]
# )
# adata_CD8.obsm["MDE_CD8+_T_Cells_scvi"] = scvi.model.utils.mde(adata_CD8.obsm[KEY_1])

adata_CD8 = ad.read_h5ad(path_to_save + "/xenium_breast_cancer_S1_R1_2_nicheVI_CD8.h5ad").copy()

del adata_CD8.uns["log1p"]

print(adata_CD8)

AnnData object with n_obs × n_vars = 16054 × 313
    obs: 'sample', '_scvi_batch', 'cell_type', '_scvi_labels', 'batch', 'predictions_scanvi', 'x', 'y', 'z', 
'observed_x', 'observed_y', 'observed_z', 'fov', 'probability', 'background', 'confusion', 'n_counts', '_indices', 
'_scvi_ind_x', 'predicted_celltype_resolvi', 'predicted_celltype_prob_resolvi', 'predictions_resolvi_proseg', 
'predictions_resolvi_proseg_coarse', 'predicted_celltype_resolvi_coarse', 'index', 'kmeans_alpha_4', 
'kmeans_alpha_5', 'kmeans_alpha_5_transferred', 'predictions_resolvi_proseg_coarse_corrected', '_scvi_sample', 
'CD8+_T_Cells_to_DCIS', 'CD8+_T_Cells_to_invasive', 'leiden_nichevi', 'leiden_scvi', 'leiden_banksy_02'
    var: 'mt'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'kmeans_alpha_5_colors', 'leiden_banksy_02', 'leiden_nichevi', 
'leiden_nichevi_colors', 'leiden_scvi', 'neighbors', 'pca', 'umap'
    obsm: 'X_banksy_02_harmony', 'X_pca', 'X_resolvi', 'X_resolvi_MDE', 'X_resolvi_proseg', 'X_scANVI', 
'X_scANVI_MDE', 'X_scanvi', 'X_spatial', 'X_umap', 'banksy_pc_20_lambda_0.0', 'banksy_pc_20_lambda_0.2', 
'celltype_transfer_resolvi', 'distance_neighbor', 'index_neighbor', 
'ln_s10_w400_scanvi_lr0.0005_poisson_X_nicheVI', 'ln_s20_w400_scanvi_lr0.0005_poisson_X_nicheVI', 
'mog_ln_s20_w400_scanvi_lr0.0005_poisson_X_nicheVI', 'neighborhood_composition', 'niche_composition', 
'niche_distances', 'niche_indexes', 'observed_spatial', 'qz1_m', 'qz1_m_niche_ct', 'spatial'
    varm: 'PCs'
    layers: 'counts', 'counts_wo_bg'
    obsp: 'connectivities', 'distances'

In [10]:
# for sample in adata.obs[setup.SAMPLE].unique().tolist()[:]:
#     # with plt.rc_context():
#     sc.pl.spatial(
#         adata_CD8[adata_CD8.obs[setup.SAMPLE] == sample],
#         spot_size=80,
#         # color=["leiden_nichevi"],
#         color=[f"{setup.NICHE}_transferred", "leiden_nichevi"],
#         # groups=[
#         #     "0",
#         #     "2",
#         # ],
#         # groups=["Stromal", "Endothelial", "Macrophages_1"],
#         ncols=1,
#         frameon=False,
#         title=sample,
#         # cmap="Set2",
#         palette=cc.glasbey_dark,
#         show=False,
#     )

In [9]:
if setup.EXPRESSION_MODEL == "scanvi":
    scvivae = scvi.model.SCANVI.load(
        dir_path=path_to_save
        + f"/{setup.EXPRESSION_MODEL}vae_E"
        + str(setup.N_EPOCHS_SCVI)
        + "_"
        + str(setup.LIKELIHOOD)
        + ".pt",
        adata=adata,
        accelerator="cpu",
    )
if setup.EXPRESSION_MODEL == "scvi":
    scvivae = scvi.model.SCVI.load(
        dir_path=path_to_save
        + f"/{setup.EXPRESSION_MODEL}vae_E"
        + str(setup.N_EPOCHS_SCVI)
        + "_"
        + str(setup.LIKELIHOOD)
        + ".pt",
        adata=adata,
        accelerator="cpu",
    )

INFO     File                                                                                                      
         ../niche-VI-experiments/checkpoints/proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2/scanvivae_E1001_poi
         sson.pt/model.pt already downloaded                                                                       


/home/nathanl/miniforge3/envs/scvi/lib/python3.12/site-packages/scvi/model/base/_save_load.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path

In [10]:
MODEL = "ln_s20_w400_scanvi_lr0.0005_poisson"

DELTA = 0.05
CORRUPTION = None
FDR = 0.05

adata.obs["leiden_nichevi"] = "Unknown"
adata.obs.loc[adata.obs[setup.CELL_TYPE] == "Endothelial", "leiden_nichevi"] = adata_CD8.obs["leiden_nichevi"]

In [11]:
adata.obs["leiden_nichevi"].value_counts()

leiden_nichevi
Unknown    262604
0            3074
1            2692
2            2666
3            2656
4            1622
5            1291
6             994
7             630
8             429
Name: count, dtype: int64

In [12]:
import nichevi
from nichevi.differential_expression import _fdr_de_prediction

model = MODEL

path_to_save_nichevae = path_to_save + "/nichevae_" + model + "_" + str(setup.N_EPOCHS_NICHEVI) + ".pt"

nichevae = nichevi.nicheSCVI.load(
    dir_path=path_to_save_nichevae,
    adata=adata,
    accelerator="cpu",
)

INFO     File                                                                                                      
         ../niche-VI-experiments/checkpoints/proseg_resolvi_scanvi_xenium_breast_cancer_S1_R1_2/nichevae_ln_s20_w40
         0_scanvi_lr0.0005_poisson_1001.pt/model.pt already downloaded                                             


/home/nathanl/miniforge3/envs/scvi/lib/python3.12/site-packages/scvi/model/base/_save_load.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load(model_path

In [13]:
scvi_DE = scvivae.differential_expression(
    adata,
    groupby="leiden_nichevi",
    group1="0",
    group2="2",
    batch_correction=False,
    delta=DELTA,
)

scvi_DE

DE...:   0%|          | 0/1 [00:00<?, ?it/s]

,proba_de,proba_not_de,bayes_factor,scale1,scale2,pseudocounts,delta,lfc_mean,lfc_median,lfc_std,...,raw_mean1,raw_mean2,non_zeros_proportion1,non_zeros_proportion2,raw_normalized_mean1,raw_normalized_mean2,is_de_fdr_0.05,comparison,group1,group2
ADH1B,0.9938,0.0062,5.076985,0.001100,0.000344,0.0,0.05,1.834255,1.850924,3.042363,...,0.072219,0.068267,0.042290,0.034509,4.177958,2.570931,True,0 vs 2,0,2
TOP2A,0.9902,0.0098,4.615524,0.000880,0.004056,0.0,0.05,-2.199496,-2.238238,1.980601,...,0.112557,1.162791,0.052049,0.222431,6.393456,40.226646,True,0 vs 2,0,2
LRRC15,0.9888,0.0112,4.480577,0.000257,0.000700,0.0,0.05,-1.614409,-1.649344,2.392387,...,0.050748,0.111778,0.030904,0.066017,3.310902,4.876233,True,0 vs 2,0,2
LYZ,0.9886,0.0114,4.462676,0.001510,0.002247,0.0,0.05,-0.628147,-0.664521,2.701928,...,0.275537,0.568267,0.095641,0.156414,14.850080,23.512506,True,0 vs 2,0,2
POSTN,0.9874,0.0126,4.361378,0.007249,0.018924,0.0,0.05,-1.490136,-1.564066,2.533978,...,1.135003,5.151913,0.297983,0.576519,64.291854,183.172543,True,0 vs 2,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF562,0.8770,0.1230,1.964323,0.003085,0.003352,0.0,0.05,-0.117028,-0.120062,0.298071,...,0.496422,0.781695,0.342225,0.451988,29.798563,32.797952,True,0 vs 2,0,2
QARS,0.8666,0.1334,1.871225,0.004400,0.004895,0.0,0.05,-0.154799,-0.157892,0.234675,...,0.750163,1.228807,0.448276,0.579895,43.823959,50.533895,True,0 vs 2,0,2
CCPG1,0.8652,0.1348,1.859168,0.006436,0.006049,0.0,0.05,0.097017,0.079830,0.307804,...,1.156148,1.562266,0.570592,0.666167,70.110039,65.745047,True,0 vs 2,0,2
C2orf42,0.8582,0.1418,1.800419,0.000596,0.000609,0.0,0.05,-0.032027,-0.033165,0.300245,...,0.094014,0.138785,0.085231,0.120405,5.814231,5.979566,True,0 vs 2,0,2


In [16]:
example_DE = nichevae.differential_expression(
    adata,
    # groupby="kmeans_alpha_5",
    groupby="leiden_nichevi",
    # groupby=setup.CELL_TYPE,
    group1="0",
    group2="2",
    batch_correction=False,
    radius=None,
    k_nn=6,
    delta=DELTA,
    count_corruption=CORRUPTION,
    niche_mode=True,
    n_samples=100,
    fdr_target=FDR,
    # weights="importance",
    pseudocounts=0.0,
)

Computing nearest neighbors...
Size of the mask matrix (dense): 26034.79 MB
Computing 6 nearest neighbors for sample 0
Computed 6 nearest neighbors for sample 0
Size of the mask matrix (dense): 13760.46 MB
Computing 6 nearest neighbors for sample 1
Computed 6 nearest neighbors for sample 1
Mean number of neighbors: 3.3 ± 2.1
Computing DE...


DE...:   0%|          | 0/1 [00:00<?, ?it/s]

Running DE for group1_group2
Running DE for group1_niche1
Running DE for niche1_group2


In [17]:
df_g1_g2 = example_DE["group1_group2"]
df_n1_g2 = example_DE["corrupted1_group2"] if CORRUPTION is not None else example_DE["niche1_group2"]
df_g1_n1 = example_DE["group1_corrupted1"] if CORRUPTION is not None else example_DE["group1_niche1"]